# 00 - Extract morphology features

This notebook is the feature-extraction workflow. It reads the supplied
FUCCI image/transcriptome HDF5 store and writes the raw and correlation-reduced morphology
matrices consumed by notebook 01. Image-processing helpers are embedded below; no source
code outside this directory is required.

**Required input**

- `inputs/fucci_final_nucmask.h5`, or set `FUCCI_H5_PATH` to the same file elsewhere.

**Generated outputs**

- `outputs/features_combined_raw.csv`: 485 QC-filtered features before correlation pruning.
- `outputs/features_combined_corrected.csv`: 277 correlation-reduced, batch-scaled features.


## Expected feature inventory

| Modality | QC-filtered raw | Correlation-reduced |
| --- | ---: | ---: |
| Nuclear fluorescence | 126 | 84 |
| Brightfield | 113 | 69 |
| Nuclear mask | 108 | 59 |
| Cell mask | 108 | 56 |
| Hybrid positioning | 30 | 9 |
| **Total** | **485** | **277** |

Assertions below make these expected outputs explicit and stop execution if the input data, package behavior, or feature schema differs.

In [ ]:
from pathlib import Path
import os

import cv2
import h5py
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from scipy.spatial.distance import squareform
from scipy.stats import entropy, hmean, mode, trim_mean
from sklearn.preprocessing import RobustScaler
from skimage.feature import local_binary_pattern
from skimage.measure import find_contours, label, regionprops
from skimage.morphology import dilation, disk
from tqdm.auto import tqdm

import cp_measure.core as cpm
import cp_measure.core.measuregranularity
import cp_measure.core.measureobjectintensitydistribution
import cp_measure.core.measureobjectsizeshape
import cp_measure.core.measuretexture

ROOT = Path.cwd().resolve()
INPUT_DIR = ROOT / "inputs"
OUTPUT_DIR = ROOT / "outputs"
CELL_DATA_PATH = Path(
    os.environ.get("FUCCI_H5_PATH", INPUT_DIR / "fucci_final_nucmask.h5")
).expanduser()
RAW_FEATURES_PATH = OUTPUT_DIR / "features_combined_raw.csv"
CORRECTED_FEATURES_PATH = OUTPUT_DIR / "features_combined_corrected.csv"
FOCAL_PLANE_INDEX = 1
CORRELATION_DISTANCE_THRESHOLD = 0.1
REQUIRED_GROUPS = ("bf", "405", "nucmask", "seg")
EXPECTED_CANDIDATE_COUNT = 533
EXPECTED_FILTERED_COUNT = 485
EXPECTED_REMOVED_COUNT = 48
EXPECTED_REDUCED_COUNT = 277

if not CELL_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Cell HDF5 file not found: {CELL_DATA_PATH}. "
        "Place it in inputs/ or set FUCCI_H5_PATH."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Embedded image-processing helpers used by this workflow.

def gini_coefficient(x):
    """Measures inequality in intensity distribution.
    Close to 1 = few bright pixels dominate (localized).
    Close to 0 = intensities more evenly spread."""
    # Ensure non-negative and non-empty
    x = np.array(x, dtype=np.float64).flatten()
    x = x[x > 0]  # remove background
    if x.size == 0:
        return 0.0
    x = np.sort(x)
    n = x.size
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * x) / np.sum(x) / n) - (n + 1) / n


def compute_texture_features(image, mask):
    """
    Compute texture features (Laplacian variance and LBP entropy) inside a binary mask.

    Parameters:
        image (np.array): Input grayscale image, typically brightfield.
        mask (np.array): Binary mask (same shape), with 0 for background and >0 for cell.

    Returns:
        lap_var (float): Variance of Laplacian inside the mask.
        lbp_entropy (float): Entropy of LBP histogram inside the mask.
    """

    # Ensure mask is binary uint8
    mask = (mask > 0).astype(np.uint8)

    # Convert image to float32 for Laplacian calculation
    if image.dtype != np.float64:
        image = image.astype(np.float64)

    # Laplacian variance: only compute over masked pixels
    laplacian = cv2.Laplacian(image, ddepth=cv2.CV_64F)
    lap_var = float(np.var(laplacian[mask > 0]))

    # LBP entropy: compute LBP and histogram over masked region
    lbp = local_binary_pattern(image, P=8, R=1, method="uniform")
    lbp_vals = lbp[mask > 0].astype(int)
    hist, _ = np.histogram(lbp_vals, bins=np.arange(0, 11), density=True)
    lbp_entropy = float(entropy(hist + 1e-10))  # Avoid log(0)

    return lap_var, lbp_entropy


def extract_basic_features(mask, name="object"):
    """
    Extract geometric features from a binary mask.

    Args:
        mask (ndarray): Binary mask of the object.
        name (str): Label for the object ("cell" or "nucleus").

    Returns:
        dict: Feature dictionary with NaN when object not found.
    """
    # Define all features upfront
    features = {
        f"{name}_valid": False,
        f"{name}_area": np.nan,
        f"{name}_perimeter": np.nan,
        f"{name}_major_axis": np.nan,
        f"{name}_minor_axis": np.nan,
        f"{name}_angle": np.nan,
        f"{name}_solidity": np.nan,
        f"{name}_eq_diameter": np.nan,
        f"{name}_aspect_ratio": np.nan,
        f"{name}_circularity": np.nan,
        f"{name}_roundness": np.nan
    }

    if mask is None or np.sum(mask) == 0:
        return features

    props = regionprops(mask.astype(np.uint8))
    if len(props) == 0:
        return features

    # Take the largest object if multiple
    prop = max(props, key=lambda p: p.area)
    area = prop.area
    perimeter = prop.perimeter if prop.perimeter > 0 else np.nan
    major = prop.major_axis_length if prop.major_axis_length > 0 else np.nan
    minor = prop.minor_axis_length if prop.minor_axis_length > 0 else np.nan
    angle = prop.orientation
    solidity = prop.solidity
    eq_diameter = prop.equivalent_diameter

    aspect_ratio = major / minor if minor and minor > 0 else np.nan
    circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter and perimeter > 0 else np.nan
    roundness = (4 * area) / (np.pi * (major ** 2)) if major and major > 0 else np.nan

    features.update({
        f"{name}_valid": True,
        f"{name}_area": area,
        f"{name}_perimeter": perimeter,
        f"{name}_major_axis": major,
        f"{name}_minor_axis": minor,
        f"{name}_angle": angle,
        f"{name}_solidity": solidity,
        f"{name}_eq_diameter": eq_diameter,
        f"{name}_aspect_ratio": aspect_ratio,
        f"{name}_circularity": circularity,
        f"{name}_roundness": roundness
    })

    return features


def nc_positioning_features(cell_mask, nucleus_mask, eps=1e-8):
    """
    Compute nuclear positioning features: distance-based features from nucleus boundary
    to cell centroid, correlation with cell boundary, statistical moments, and gini.

    If cell_mask or nucleus_mask are invalid, returns all features as np.nan.
    """
    # List of keys to always return
    feature_keys = [
        "mean_Rprime", "median_Rprime", "mode_Rprime", "max_Rprime", "min_Rprime",
        "p25_Rprime", "p75_Rprime", "std_Rprime", "harmonic_mean_Rprime",
        "trimmed_mean_3_Rprime", "trimmed_mean_5_Rprime", "trimmed_mean_15_Rprime",
        "trimmed_mean_25_Rprime", "mad_Rprime", "range_Rprime", "iqr_Rprime",
        "S_Rprime2", "S_Rprime3", "mean_Rprime2", "mean_Rprime3",
        "mean_Rprime4", "mean_Rprime5", "S_abs_dev2", "S_abs_dev3",
        "mean_abs_dev2", "mean_abs_dev3", "mean_abs_dev4", "mean_abs_dev5",
        "num_peaks_Rprime", "gini_Rprime", "corr_Rprime_cell"
    ]

    # Initialize all features with np.nan
    feats = {k: np.nan for k in feature_keys}

    # Validate masks
    if cell_mask is None or nucleus_mask is None or np.sum(cell_mask) == 0 or np.sum(nucleus_mask) == 0:
        return feats

    try:
        # --- 1. Cell centroid ---
        cell_props = regionprops(label(cell_mask.astype(int)))
        if len(cell_props) == 0:
            return feats
        cell_centroid = cell_props[0].centroid  # (row, col)

        # --- 2. Nucleus boundary ---
        contours = find_contours(nucleus_mask, 0.5)
        if len(contours) == 0:
            return feats
        nuc_contour = max(contours, key=lambda x: x.shape[0])
        ys, xs = nuc_contour[:,0], nuc_contour[:,1]
        xs_smooth = gaussian_filter1d(xs, sigma=1)
        ys_smooth = gaussian_filter1d(ys, sigma=1)

        # --- 3. Distances R' from nucleus boundary to cell centroid ---
        R_prime = np.sqrt((xs_smooth - cell_centroid[1])**2 + (ys_smooth - cell_centroid[0])**2)

        # --- 4. Compute statistical features ---
        feats["mean_Rprime"] = np.mean(R_prime)
        feats["median_Rprime"] = np.median(R_prime)
        feats["mode_Rprime"] = mode(R_prime, keepdims=True)[0][0]
        feats["max_Rprime"] = np.max(R_prime)
        feats["min_Rprime"] = np.min(R_prime)
        feats["p25_Rprime"] = np.percentile(R_prime, 25)
        feats["p75_Rprime"] = np.percentile(R_prime, 75)
        feats["std_Rprime"] = np.std(R_prime)
        feats["harmonic_mean_Rprime"] = hmean(R_prime + eps)
        feats["trimmed_mean_3_Rprime"] = trim_mean(R_prime, 0.03)
        feats["trimmed_mean_5_Rprime"] = trim_mean(R_prime, 0.05)
        feats["trimmed_mean_15_Rprime"] = trim_mean(R_prime, 0.15)
        feats["trimmed_mean_25_Rprime"] = trim_mean(R_prime, 0.25)
        feats["mad_Rprime"] = np.mean(np.abs(R_prime - feats["mean_Rprime"]))
        feats["range_Rprime"] = feats["max_Rprime"] - feats["min_Rprime"]
        feats["iqr_Rprime"] = feats["p75_Rprime"] - feats["p25_Rprime"]

        # Higher-order moments
        feats["S_Rprime2"] = np.mean(R_prime**2)
        feats["S_Rprime3"] = np.mean(R_prime**3)
        feats["mean_Rprime2"] = feats["S_Rprime2"]
        feats["mean_Rprime3"] = feats["S_Rprime3"]
        feats["mean_Rprime4"] = np.mean(R_prime**4)
        feats["mean_Rprime5"] = np.mean(R_prime**5)

        abs_dev = np.abs(R_prime - feats["mean_Rprime"])
        feats["S_abs_dev2"] = np.mean(abs_dev**2)
        feats["S_abs_dev3"] = np.mean(abs_dev**3)
        feats["mean_abs_dev2"] = np.mean(abs_dev**2)
        feats["mean_abs_dev3"] = np.mean(abs_dev**3)
        feats["mean_abs_dev4"] = np.mean(abs_dev**4)
        feats["mean_abs_dev5"] = np.mean(abs_dev**5)

        # Peaks
        peaks, _ = find_peaks(R_prime)
        feats["num_peaks_Rprime"] = len(peaks)

        # Gini coefficient
        sorted_R = np.sort(R_prime)
        n = len(sorted_R)
        cumR = np.cumsum(sorted_R)
        feats["gini_Rprime"] = (n + 1 - 2 * np.sum(cumR) / cumR[-1]) / n

        # Correlation with cell boundary signature
        cell_contours = find_contours(cell_mask, 0.5)
        cell_contour = max(cell_contours, key=lambda x: x.shape[0])
        ys_cell, xs_cell = cell_contour[:,0], cell_contour[:,1]
        xs_cell_smooth = gaussian_filter1d(xs_cell, sigma=1)
        ys_cell_smooth = gaussian_filter1d(ys_cell, sigma=1)
        R_cell = np.sqrt((xs_cell_smooth - cell_centroid[1])**2 + (ys_cell_smooth - cell_centroid[0])**2)
        min_len = min(len(R_cell), len(R_prime))
        feats["corr_Rprime_cell"] = np.corrcoef(R_prime[:min_len], R_cell[:min_len])[0,1]

    except Exception:
        # Any error → keep np.nan
        pass

    return feats


def measure_cp(image, mask):
    binary_mask = np.round(mask / 255.0).astype(np.uint8)
    scaled_image = image.astype(float) / 65535.0
    features = {}
    features.update(
        cpm.measureobjectintensitydistribution.get_radial_zernikes(
            binary_mask, scaled_image
        )
    )
    features.update(cpm.measureobjectsizeshape.get_sizeshape(binary_mask, scaled_image))
    features.update(cpm.measureobjectsizeshape.get_zernike(binary_mask, scaled_image))
    features.update(cpm.measureobjectsizeshape.get_feret(binary_mask, scaled_image))
    features.update(cpm.measuregranularity.get_granularity(binary_mask, scaled_image))
    features.update(cpm.measuretexture.get_texture(binary_mask, scaled_image))
    return {name: value[0] for name, value in features.items()}


def mask_features(mask, prefix):
    features = extract_basic_features(mask)
    features = {
        name.replace("object_", f"{prefix}_fbasic_"): value
        for name, value in features.items()
    }
    features.pop(f"{prefix}_fbasic_valid", None)
    return features


def nuclear_positioning_features(cell_mask, nuclear_mask):
    features = nc_positioning_features(cell_mask, nuclear_mask)
    return {f"fnpos_{name}": value for name, value in features.items()}


def texture_features(image, mask, prefix):
    laplacian_variance, laplacian_entropy = (
        compute_texture_features(image, mask)
    )
    return {
        f"{prefix}_txtlap_var": laplacian_variance,
        f"{prefix}_txtlap_entrpy": laplacian_entropy,
    }


def gini_feature(mask, image, prefix, dilation_radius=3):
    enlarged_mask = dilation(mask > 0, disk(dilation_radius))
    values = image[enlarged_mask]
    value = float(gini_coefficient(values)) if values.size else None
    return {f"{prefix}_gini": value}


def custom_features(image_405, image_bf, nuclear_mask, cell_mask):
    feature_blocks = (
        mask_features(nuclear_mask, "nmask"),
        mask_features(cell_mask, "cmask"),
        nuclear_positioning_features(cell_mask, nuclear_mask),
        texture_features(image_405, nuclear_mask, "405"),
        texture_features(image_bf, cell_mask, "bf"),
        gini_feature(cell_mask, image_bf, "bf"),
        gini_feature(nuclear_mask, image_405, "405"),
    )
    features = {}
    for block in feature_blocks:
        features.update(block)
    return features


IMAGE_FEATURE_FAMILIES = {
    "AngularSecondMoment", "Contrast", "Correlation",
    "DifferenceEntropy", "DifferenceVariance", "Entropy",
    "Granularity", "InfoMeas1", "InfoMeas2",
    "InverseDifferenceMoment", "RadialDistribution", "SumAverage",
    "SumEntropy", "SumVariance", "Variance", "gini", "txtlap",
}


def feature_modality(feature_name):
    if feature_name.startswith("fnpos_"):
        return "hybrid positioning"
    if feature_name.startswith("nmask_"):
        return "nuclear mask"
    if feature_name.startswith("cmask_"):
        return "cell mask"
    for channel, image_modality, mask_modality in (
        ("405_", "nuclear fluorescence", "nuclear mask"),
        ("bf_", "brightfield", "cell mask"),
    ):
        if feature_name.startswith(channel):
            family = feature_name[len(channel):].split("_", 1)[0]
            return image_modality if family in IMAGE_FEATURE_FAMILIES else mask_modality
    raise ValueError(f"Unclassified feature: {feature_name}")


def modality_counts(columns):
    return pd.Series([feature_modality(column) for column in columns]).value_counts().to_dict()


## Image and mask feature extraction

In [ ]:
cp_by_cell = {}
custom_by_cell = {}
batch_by_cell = {}
skip_counts = {"missing required group": 0, "empty mask": 0}

with h5py.File(CELL_DATA_PATH, "r") as h5_file:
    cell_ids = list(h5_file.keys())
    if not cell_ids:
        raise ValueError("The cell HDF5 file contains no root-level cell groups.")

    for cell_id in tqdm(cell_ids, desc="Extracting features"):
        cell_group = h5_file[cell_id]
        if "exp_id" not in cell_group.attrs:
            raise ValueError(f"Cell {cell_id} has no exp_id attribute.")
        if not all(name in cell_group for name in REQUIRED_GROUPS):
            skip_counts["missing required group"] += 1
            continue

        plane_keys = list(cell_group["405"].keys())
        if len(plane_keys) <= FOCAL_PLANE_INDEX:
            raise ValueError(
                f"Cell {cell_id} has fewer than {FOCAL_PLANE_INDEX + 1} 405 planes."
            )
        focal_plane = plane_keys[FOCAL_PLANE_INDEX]
        missing_plane = [
            name for name in REQUIRED_GROUPS if focal_plane not in cell_group[name]
        ]
        if missing_plane:
            raise ValueError(
                f"Cell {cell_id}, plane {focal_plane}, is missing from {missing_plane}."
            )

        image_405 = cell_group["405"][focal_plane][()]
        image_bf = cell_group["bf"][focal_plane][()]
        nuclear_mask = cell_group["nucmask"][focal_plane][()]
        cell_mask = cell_group["seg"][focal_plane][()]
        arrays = (image_405, image_bf, nuclear_mask, cell_mask)

        if any(array.ndim != 2 for array in arrays):
            raise ValueError(f"Cell {cell_id}, plane {focal_plane}, has a non-2D array.")
        if len({array.shape for array in arrays}) != 1:
            raise ValueError(f"Cell {cell_id}, plane {focal_plane}, has mismatched shapes.")
        for mask_name, mask in (("nucmask", nuclear_mask), ("seg", cell_mask)):
            if not np.isin(np.unique(mask), (0, 255)).all():
                raise ValueError(
                    f"Cell {cell_id} {mask_name} is not encoded exclusively as 0/255."
                )

        if np.all(nuclear_mask == 0) or np.all(cell_mask == 0):
            skip_counts["empty mask"] += 1
            continue

        cp_features = {}
        cp_features.update(
            {f"405_{name}": value for name, value in measure_cp(image_405, nuclear_mask).items()}
        )
        cp_features.update(
            {f"bf_{name}": value for name, value in measure_cp(image_bf, cell_mask).items()}
        )
        cp_by_cell[cell_id] = cp_features
        custom_by_cell[cell_id] = custom_features(
            image_405, image_bf, nuclear_mask, cell_mask
        )
        batch_by_cell[cell_id] = cell_group.attrs["exp_id"]

if not cp_by_cell:
    raise ValueError("No cells passed the required-group and non-empty-mask checks.")
print(f"Processed {len(cp_by_cell)} cells; skipped {skip_counts}.")


In [ ]:
df_cp = pd.DataFrame.from_dict(cp_by_cell, orient="index")
df_custom = pd.DataFrame.from_dict(custom_by_cell, orient="index")
if not df_cp.index.equals(df_custom.index):
    raise ValueError("CellProfiler and custom feature rows are not aligned.")

candidate_features = df_cp.join(df_custom)
if candidate_features.columns.duplicated().any():
    duplicates = candidate_features.columns[candidate_features.columns.duplicated()].tolist()
    raise ValueError(f"Duplicate candidate feature names: {duplicates}")
if candidate_features.shape[1] != EXPECTED_CANDIDATE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_CANDIDATE_COUNT} candidates, got {candidate_features.shape[1]}."
    )
print(f"Candidate feature matrix: {candidate_features.shape}")


## Quality control and raw-feature export

The implemented filtering criterion removes a feature when its number of distinct non-missing values is less than half the number of retained cells. An all-missing-column check is then applied as a safeguard. On the source data these rules remove 48 of 533 candidates, leaving 485 raw features.

After column filtering, missing values and both positive and negative infinity are replaced with zero, preserving the source behavior. The resulting 485-column matrix is exported before correlation reduction.

In [ ]:
distinct_counts = candidate_features.nunique()
low_information_columns = distinct_counts[
    distinct_counts < (len(candidate_features) / 2)
].index
features_filtered = candidate_features.drop(columns=low_information_columns)
all_missing_columns = features_filtered.columns[features_filtered.isna().all()]
features_filtered = features_filtered.drop(columns=all_missing_columns)
removed_columns = list(low_information_columns) + list(all_missing_columns)

if len(removed_columns) != EXPECTED_REMOVED_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_REMOVED_COUNT} filtered columns, got {len(removed_columns)}."
    )
features_filtered = features_filtered.fillna(0)
features_filtered = features_filtered.replace([np.inf, -np.inf], 0)
if features_filtered.shape[1] != EXPECTED_FILTERED_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FILTERED_COUNT} raw features, got {features_filtered.shape[1]}."
    )
if not np.isfinite(features_filtered.to_numpy(dtype=float)).all():
    raise ValueError("The filtered feature matrix still contains non-finite values.")

expected_filtered_modalities = {
    "nuclear fluorescence": 126,
    "brightfield": 113,
    "nuclear mask": 108,
    "cell mask": 108,
    "hybrid positioning": 30,
}
observed_filtered_modalities = modality_counts(features_filtered.columns)
if observed_filtered_modalities != expected_filtered_modalities:
    raise ValueError(
        f"Unexpected raw modality counts: {observed_filtered_modalities}"
    )
features_filtered.to_csv(RAW_FEATURES_PATH, index=True, header=True)
print(f"Exported {features_filtered.shape[1]} raw features to {RAW_FEATURES_PATH}.")
print(f"Raw modality counts: {observed_filtered_modalities}")


## Correlation reduction

Feature-feature Spearman correlations are converted to absolute values, with any correlation-matrix missing values filled with zero. Distance is defined as `1 - abs(Spearman rho)`. Ward hierarchical linkage is applied to the condensed distance matrix, and clusters are cut at distance threshold **0.1**. 

One representative is retained per cluster: the first feature encountered in the existing input column order. 

In [ ]:
absolute_spearman = features_filtered.corr(method="spearman").abs().fillna(0)
distance_matrix = 1 - absolute_spearman
linkage_matrix = linkage(
    squareform(distance_matrix, checks=False),
    method="ward",
)
cluster_labels = fcluster(
    linkage_matrix,
    t=CORRELATION_DISTANCE_THRESHOLD,
    criterion="distance",
)
feature_clusters = pd.DataFrame(
    {"Feature": features_filtered.columns, "Cluster": cluster_labels}
)
representative_features = (
    feature_clusters.groupby("Cluster")["Feature"].first().tolist()
)
features_reduced = features_filtered[representative_features].copy()

if features_reduced.shape[1] != EXPECTED_REDUCED_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_REDUCED_COUNT} reduced features, got {features_reduced.shape[1]}."
    )
expected_reduced_modalities = {
    "nuclear fluorescence": 84,
    "brightfield": 69,
    "nuclear mask": 59,
    "cell mask": 56,
    "hybrid positioning": 9,
}
observed_reduced_modalities = modality_counts(features_reduced.columns)
if observed_reduced_modalities != expected_reduced_modalities:
    raise ValueError(
        f"Unexpected reduced modality counts: {observed_reduced_modalities}"
    )
print(f"Correlation reduction: {features_filtered.shape[1]} -> {features_reduced.shape[1]}.")
print(f"Reduced modality counts: {observed_reduced_modalities}")


## Batch metadata and transductive preprocessing

Each retained cell is assigned its HDF5 batch id `exp_id`. All 277 reduced features are independently transformed within each `exp_id` batch using scikit-learn `RobustScaler` with centering and scaling enabled (batch median and interquartile range). No feature family is excluded and no global scaler is fitted.

In [ ]:
batch_metadata = pd.Series(batch_by_cell, name="exp_id").reindex(features_reduced.index)
if batch_metadata.isna().any():
    missing_ids = batch_metadata.index[batch_metadata.isna()].tolist()
    raise ValueError(f"Missing exp_id metadata for retained cells: {missing_ids[:10]}")

features_with_batch = features_reduced.copy()
features_with_batch["exp_id"] = batch_metadata
scaled_batches = []
for _, batch in features_with_batch.groupby("exp_id"):
    batch_features = batch[features_reduced.columns]
    scaler = RobustScaler(with_centering=True, with_scaling=True)
    scaled_batches.append(
        pd.DataFrame(
            scaler.fit_transform(batch_features),
            index=batch_features.index,
            columns=features_reduced.columns,
        )
    )

features_final = pd.concat(scaled_batches, axis=0)
if set(features_final.index) != set(features_reduced.index):
    raise ValueError("Batch scaling changed the retained cell set.")
if list(features_final.columns) != list(features_reduced.columns):
    raise ValueError("Batch scaling changed the selected feature schema.")


## Final export

The final matrix contains the correlation-reduced features after within-batch robust scaling. 

In [ ]:
if features_final.shape != (len(features_reduced), EXPECTED_REDUCED_COUNT):
    raise ValueError(f"Unexpected final matrix shape: {features_final.shape}")
if not np.isfinite(features_final.to_numpy(dtype=float)).all():
    raise ValueError("The final feature matrix contains non-finite values.")
if modality_counts(features_final.columns) != expected_reduced_modalities:
    raise ValueError("The final export has unexpected modality counts.")

features_final.to_csv(CORRECTED_FEATURES_PATH, index=True, header=True)
print(
    f"Exported {features_final.shape[0]} cells x {features_final.shape[1]} features "
    f"to {CORRECTED_FEATURES_PATH}."
)
